# Uji Signifikansi Statistik — Diebold–Mariano (Multivariate vs Univariate)

## Analisis Prediksi IHSG Menggunakan Multivariate LSTM Berdasarkan Data Historis dan Variabel Makroekonomi

Notebook ini menjawab bagian dari **Rumusan Masalah 2**: apakah perbedaan kinerja prediksi antara model **multivariate dual-branch LSTM** dan **baseline univariate** bersifat **signifikan secara statistik**, bukan sekadar lebih kecil secara numerik.

Metrik seperti RMSE/MAPE hanya menunjukkan selisih angka. Uji **Diebold–Mariano (DM)** (Diebold & Mariano, 1995) menguji secara formal apakah dua model memiliki akurasi prediksi yang sama. Uji ini bersifat *post-hoc*: notebook ini **hanya membaca prediksi yang sudah tersimpan** dari pipeline pemodelan (`model/model_multivariate_v3/predictions_test.joblib`) dan **tidak melatih ulang model apa pun**, sehingga tidak mengubah hasil yang sudah ada.


## 1. Imports

In [1]:
import numpy as np
import joblib
from scipy.stats import t as tdist
import pandas as pd

## 2. Muat Prediksi Test Set

Memuat prediksi kedua model pada periode pengujian yang sama (nilai aktual, prediksi univariate, dan prediksi multivariate) dalam skala IHSG asli.

In [2]:
pred = joblib.load("../model/model_multivariate_v3/predictions_test.joblib")

y_true  = np.asarray(pred["multivariate"]["y_true"]).flatten().astype(float)
y_multi = np.asarray(pred["multivariate"]["y_pred"]).flatten().astype(float)
y_uni   = np.asarray(pred["univariate"]["y_pred"]).flatten().astype(float)

N = len(y_true)
rmse = lambda yt, yp: float(np.sqrt(np.mean((yt - yp) ** 2)))
print(f"Jumlah observasi test: {N}")
print(f"RMSE univariate   : {rmse(y_true, y_uni):.2f}")
print(f"RMSE multivariate : {rmse(y_true, y_multi):.2f}")

Jumlah observasi test: 485
RMSE univariate   : 129.05
RMSE multivariate : 122.89


## 3. Galat Prediksi (Forecast Error)

Galat tiap model dihitung sebagai selisih nilai aktual dan prediksi, `e = y_aktual − y_prediksi`, untuk seluruh titik pada periode test.

In [3]:
e_uni   = y_true - y_uni     # galat model univariate (baseline)
e_multi = y_true - y_multi   # galat model multivariate dual-branch

## 4. Fungsi Uji Diebold–Mariano

Uji DM bekerja pada **selisih fungsi loss (loss differential)** antara dua model:

$$d_t = g(e_{1,t}) - g(e_{2,t})$$

dengan $g(\cdot)$ fungsi loss (di sini *squared error*, $g(e)=e^2$). Hipotesis:

- $H_0$: kedua model memiliki akurasi prediksi yang sama ($E[d_t]=0$)
- $H_1$: akurasi kedua model berbeda

Statistik DM adalah rata-rata $\bar{d}$ dibagi akar ragam jangka panjangnya. Untuk prediksi satu langkah ke depan ($h=1$) tidak ada suku autokovariansi. Diterapkan pula koreksi sampel kecil **Harvey–Leybourne–Newbold (1997)** dan nilai-$p$ dihitung dari distribusi $t$ dengan $T-1$ derajat bebas. Konvensi: model 1 = univariate, model 2 = multivariate, sehingga **DM positif berarti multivariate lebih akurat**.

In [4]:
def dm_test(e1, e2, h=1, power=2):
    """Diebold-Mariano. e1=univariate, e2=multivariate.
    power=2 -> squared-error loss; power=1 -> absolute-error loss.
    DM>0 => model 2 (multivariate) lebih akurat."""
    e1, e2 = np.asarray(e1), np.asarray(e2)
    d = np.abs(e1) ** power - np.abs(e2) ** power
    T = len(d); dbar = d.mean()
    gamma0 = np.sum((d - dbar) ** 2) / T
    acov = sum(2 * np.sum((d[k:] - dbar) * (d[:-k] - dbar)) / T for k in range(1, h))
    var_dbar = (gamma0 + acov) / T
    DM = dbar / np.sqrt(var_dbar)
    k_adj = np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)   # Harvey-Leybourne-Newbold
    DM_hln = DM * k_adj
    p_value = 2 * tdist.cdf(-abs(DM_hln), df=T - 1)
    return {"DM": float(DM), "DM_hln": float(DM_hln), "p_value": float(p_value),
            "dbar": float(dbar), "T": int(T)}

## 5. Hasil Uji — Squared-Error Loss

Loss utama yang digunakan adalah *squared error*, konsisten dengan metrik RMSE.

In [5]:
res_sq = dm_test(e_uni, e_multi, h=1, power=2)
print("Statistik DM (HLN) :", round(res_sq["DM_hln"], 4))
print("p-value (dua sisi) :", round(res_sq["p_value"], 4))
print("Rata-rata d (loss_uni - loss_multi):", round(res_sq["dbar"], 4))
arah = "multivariate" if res_sq["dbar"] > 0 else "univariate"
sig  = "signifikan (tolak H0)" if res_sq["p_value"] < 0.05 else "tidak signifikan (gagal tolak H0)"
print(f"\nKesimpulan (alpha=0.05): perbedaan akurasi {sig}; arah keunggulan: {arah}.")

Statistik DM (HLN) : 1.7406
p-value (dua sisi) : 0.0824
Rata-rata d (loss_uni - loss_multi): 1553.5143

Kesimpulan (alpha=0.05): perbedaan akurasi tidak signifikan (gagal tolak H0); arah keunggulan: multivariate.


## 6. Robustness — Absolute-Error Loss

Sebagai pemeriksaan ketahanan, uji diulang dengan loss *absolute error* (sejalan dengan MAPE/MAE).

In [6]:
res_ab = dm_test(e_uni, e_multi, h=1, power=1)
print("Statistik DM (HLN) :", round(res_ab["DM_hln"], 4))
print("p-value (dua sisi) :", round(res_ab["p_value"], 4))

Statistik DM (HLN) : 1.8625
p-value (dua sisi) : 0.0631


## 7. Ringkasan & Export

Tabel ringkas disimpan ke `reports/figures_v3/dm_test_result.csv` sebagai artefak pelengkap (file baru; tidak menimpa hasil lain).

In [7]:
summary = pd.DataFrame([
    {"loss": "Squared error", "DM_stat_HLN": round(res_sq["DM_hln"], 4),
     "p_value": round(res_sq["p_value"], 4),
     "signifikan_5pct": res_sq["p_value"] < 0.05},
    {"loss": "Absolute error", "DM_stat_HLN": round(res_ab["DM_hln"], 4),
     "p_value": round(res_ab["p_value"], 4),
     "signifikan_5pct": res_ab["p_value"] < 0.05},
])
summary.to_csv("../reports/figures_v3/dm_test_result.csv", index=False)
summary

,loss,DM_stat_HLN,p_value,signifikan_5pct
0,Squared error,1.7406,0.0824,False
1,Absolute error,1.8625,0.0631,False


## 8. Interpretasi

Uji Diebold–Mariano memberikan dasar statistik (bukan sekadar perbandingan angka RMSE/MAPE) untuk menilai apakah penambahan variabel makroekonomi pada arsitektur dual-branch benar-benar mengubah akurasi prediksi IHSG secara bermakna. Nilai-$p$ di bawah 0,05 menandakan perbedaan akurasi yang signifikan secara statistik; arah keunggulan ditentukan oleh tanda rata-rata loss differential.

> Catatan metode: uji DM mengacu pada Diebold & Mariano (1995), dengan koreksi sampel kecil Harvey, Leybourne & Newbold (1997). Pendekatan serupa (uji Diebold–Mariano untuk membandingkan akurasi prediksi) juga digunakan oleh Rosinus & Lánský (2025).